# Phase B v2: FinBERT (title-only, clean)

**목적**: v4 파일의 **title이 실제로 존재하는 41,588행**에 대해서만 FinBERT 감성 점수 계산.

**이전 버전(Phase_B_FinBERT.ipynb) 문제점**
1. 전체 1,183만 행 처리 → 99.6%가 빈 title
2. 빈 title을 `'neutral news'`로 치환 → mean≈0.028의 가짜 점수 양산
3. V2Tone severity와 가중합 → orthogonal(독립) 분석 불가

**이번 버전**
- 입력: `finbert_input.parquet` (1.7 MB, event_id + title, 41,588행)
- 모델: `ProsusAI/finbert` (GPU 배치 추론)
- 출력: `finbert_results.parquet` (event_id, finbert_neg, finbert_neu, finbert_pos, finbert_score)
  - `finbert_score = pos - neg ∈ [-1, +1]` (양수=긍정, 음수=부정)
- 후속 merge 시: v4의 기존 finbert 컬럼은 **모두 NULL로 리셋**하고 이 41K행만 업데이트 (v5)

**예상 런타임**: T4 GPU 기준 약 2~5분 (41K × batch=64)

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/nabi_hyoghaw'  # ← 본인 경로로 수정
os.chdir(PROJECT_DIR)
print('cwd:', os.getcwd())

In [ ]:
!pip install -q transformers torch pandas pyarrow tqdm

import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. 입력 로드

**중요**: `finbert_input.parquet`는 로컬에서 `scripts/finbert_extract_input.py`로 생성된 파일입니다. Google Drive의 `data/processed/` 안에 있어야 합니다.

In [ ]:
import pandas as pd

INPUT_PATH = 'data/processed/finbert_input.parquet'
df = pd.read_parquet(INPUT_PATH)
print(f'rows       : {len(df):,}')
print(f'columns    : {df.columns.tolist()}')
print(f'title NULL : {df["title"].isna().sum()}')
print(f'title len  : min={df["title"].str.len().min()}, avg={df["title"].str.len().mean():.0f}, max={df["title"].str.len().max()}')
df.head(5)

## 2. FinBERT 모델 로드

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = 'ProsusAI/finbert'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Label order: ProsusAI/finbert -> 0=positive, 1=negative, 2=neutral  (주의!)
print('id2label:', model.config.id2label)

## 3. 배치 추론

**주의**: ProsusAI/finbert의 라벨 순서는 `0=positive, 1=negative, 2=neutral` 입니다. softmax 출력 인덱스를 `id2label`에서 동적으로 가져와서 혼동을 방지합니다.

In [ ]:
import numpy as np
from tqdm.auto import tqdm

BATCH_SIZE = 64
MAX_LEN    = 256

# Label 인덱스를 id2label에서 동적으로 찾기 (모델이 바뀌어도 안전)
id2label = {i: v.lower() for i, v in model.config.id2label.items()}
pos_idx = [i for i, v in id2label.items() if v == 'positive'][0]
neg_idx = [i for i, v in id2label.items() if v == 'negative'][0]
neu_idx = [i for i, v in id2label.items() if v == 'neutral'][0]
print(f'pos_idx={pos_idx}, neg_idx={neg_idx}, neu_idx={neu_idx}')

texts = df['title'].astype(str).tolist()
n = len(texts)
neg = np.zeros(n, dtype=np.float32)
neu = np.zeros(n, dtype=np.float32)
pos = np.zeros(n, dtype=np.float32)

for i in tqdm(range(0, n, BATCH_SIZE), desc='FinBERT'):
    batch = texts[i:i+BATCH_SIZE]
    inputs = tokenizer(
        batch, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    neg[i:i+len(batch)] = probs[:, neg_idx]
    neu[i:i+len(batch)] = probs[:, neu_idx]
    pos[i:i+len(batch)] = probs[:, pos_idx]

df['finbert_neg']   = neg
df['finbert_neu']   = neu
df['finbert_pos']   = pos
df['finbert_score'] = (pos - neg).astype(np.float32)  # [-1, +1]

print('\n=== distribution ===')
print(df[['finbert_neg','finbert_neu','finbert_pos','finbert_score']].describe())
print('\nsign buckets:')
print(f'  negative (< -0.3): {(df.finbert_score < -0.3).sum():,}')
print(f'  neutral  [-0.3,0.3]: {df.finbert_score.between(-0.3,0.3).sum():,}')
print(f'  positive (>  0.3): {(df.finbert_score >  0.3).sum():,}')

## 4. Sanity check (샘플 확인)

무작위 샘플 10건을 출력해 부정/중립/긍정 분류가 직관과 맞는지 확인.

In [ ]:
print('=== most NEGATIVE ===')
for _, r in df.nsmallest(5, 'finbert_score').iterrows():
    print(f'  {r.finbert_score:+.3f} | {r.title[:90]}')
print('\n=== most POSITIVE ===')
for _, r in df.nlargest(5, 'finbert_score').iterrows():
    print(f'  {r.finbert_score:+.3f} | {r.title[:90]}')
print('\n=== random sample ===')
for _, r in df.sample(5, random_state=42).iterrows():
    print(f'  {r.finbert_score:+.3f} | {r.title[:90]}')

## 5. 저장

결과만 저장 (title 제외, event_id + 점수). 로컬에서 merge 스크립트가 v4와 조인해 v5를 만듭니다.

In [ ]:
OUT_PATH = 'data/processed/finbert_results.parquet'
df[['event_id','finbert_neg','finbert_neu','finbert_pos','finbert_score']].to_parquet(
    OUT_PATH, index=False, compression='zstd'
)
import os
print(f'Saved: {OUT_PATH}')
print(f'Size : {os.path.getsize(OUT_PATH)/1024:.1f} KB')
print(f'Rows : {len(df):,}')
print('\n다음 단계: 로컬에서 scripts/finbert_merge_to_v5.py 실행')